# Schema — add `is_milestone` to the task layerAdds a boolean (`esriFieldTypeSmallInteger`) field to the task layer in`datateam_portfolio_v2` so milestones can be flagged as a first-classvisual treatment in the app.**Run this once, before deploying the milestones release.** Idempotent —re-running is a no-op once the field exists.You must be **owner or admin** of `datateam_portfolio_v2`.**Field shape**- Name: `is_milestone`- Type: `esriFieldTypeSmallInteger` (1 = milestone, 0 / NULL = regular task)- Nullable: yes (existing records will be NULL until flagged via the in-app form or the console-command helper)After this notebook runs, the app's `agolTaskToLocal` reads `is_milestone`gracefully (NULL / missing → false), and the new task-form checkbox canpersist a value.

In [ ]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer

gis = GIS("home")
print(f"Signed in as {gis.users.me.username} @ {gis.url}")

TASK_LAYER_URL = "https://services3.arcgis.com/9coHY2fvuFjG9HQX/ArcGIS/rest/services/datateam_portfolio_v2/FeatureServer/1"
tasks = FeatureLayer(TASK_LAYER_URL, gis)
print("Task layer:", tasks.properties.get('name', '(unknown)'))

## Step 1 — Check whether the field already existsIdempotency check. If `is_milestone` is already in the schema, this notebookhas been run before — skip ahead to verification.

In [ ]:
existing_fields = [f['name'] for f in tasks.properties.fields]
print(f"Total fields: {len(existing_fields)}")
print()
already_exists = 'is_milestone' in existing_fields
if already_exists:
    print("✓ is_milestone is already in the schema — nothing to add.")
else:
    print("ℹ is_milestone NOT in the schema — Step 2 will add it.")

## Step 2 — Add the fieldOnly runs if the field doesn't exist. Writes to the layer's `addToDefinition`endpoint.

In [ ]:
if already_exists:
    print("Skipped — already present.")
else:
    field_def = {
        "name": "is_milestone",
        "type": "esriFieldTypeSmallInteger",
        "alias": "Is Milestone",
        "nullable": True,
        "editable": True,
        "defaultValue": None
    }
    result = tasks.manager.add_to_definition({"fields": [field_def]})
    print("add_to_definition result:", result)

## Step 3 — VerifyRe-reads the layer and confirms the field is present. Also queries to confirmthe field is reachable (returns 0 because no record has been flagged yet).

In [ ]:
tasks_fresh = FeatureLayer(TASK_LAYER_URL, gis)
fields_now = [f['name'] for f in tasks_fresh.properties.fields]
print(f"Field count: {len(fields_now)}")
print()
if 'is_milestone' in fields_now:
    print("✓ is_milestone is in the schema.")
    sample = tasks_fresh.query(where="is_milestone = 1", out_fields="OBJECTID", return_count_only=True)
    print(f"  Tasks currently flagged as milestones: {sample}")
    print()
    print("Next: deploy the app code (v1.73.0.0) and run the console-command")
    print("migration helper in the browser to flag any existing milestone-like tasks.")
else:
    print("✗ is_milestone NOT in the schema — check the Step 2 result for errors.")

## Rollback (if needed)To remove the field again — for example if a deploy needs to be rolled back:    tasks.manager.delete_from_definition({"fields": [{"name": "is_milestone"}]})This deletes the column and any data in it. Don't run it casually.